# Projet — Exploitation du dateset CL-Drive

## Estimation de la charge cognitive du conducteur

Ce TP vient après les TD1, TD2, TD3 et TD4.

Les TD ont déjà permis de travailler :

- la compréhension du papier CL-Drive ;
- le protocole expérimental ;
- la segmentation en fenêtres de 10 s ;
- le prétraitement EEG ;
- l'extraction des features EEG.

Le point de départ du TP est donc le dossier généré à la fin du TD4 :

```text
EEG_Features_10s/
```

Ce TP ne revient pas sur le calcul des features. Il exploite les features déjà extraites pour construire un pipeline d'apprentissage automatique.

## Objectif du TP

Construire un pipeline complet :

```text
EEG_Features_10s
→ Normalized_Features_10s/EEG
→ Normalized_Features_10s_With_Label/EEG
→ Dataset EEG supervisé
→ Classification de la charge cognitive
→ Évaluation
→ Interprétation
```

Dans un premier temps, on se limite à l'EEG uniquement.

La multimodalité, c'est-à-dire l'ajout de ECG, EDA et Gaze, sera proposée uniquement comme extension à la fin du sujet.

## 1. Structure attendue des dossiers

Avant de commencer, le dossier de travail doit contenir au minimum :

```text
Data/
├── EEG/ID_x
│   ├── ... fichiers level_1, level_2, ..., level_9
│   ├── ... fichiers baseline
│   └── ... fichiers filtered_*
│
├── EEG_Features_10s/
│   ├── ID1_EEG_features.csv
│   ├── ID2_EEG_features.csv
│   └── ...
│
├── Labels/
│   ├── ID1.csv
│   ├── ID2.csv
│   └── ...
```

Le TP va générer deux nouveaux dossiers :

```text
Data/
├── Normalized_Features_10s/
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
│
├── Normalized_Features_10s_With_Label/(avec colonnes Level et Label)
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
```

## Question 

Pourquoi ne faut-il pas entraîner directement les modèles sur les fichiers `EEG_Features_10s`  ?

### Réponse 

- Fuite de données (data leakage) : utiliser statistiques/globales calculées sur tout EEG_Features_10s (ou inclure colonnes liées aux labels) permettrait au modèle d’exploiter des informations du jeu de test.
- Variabilité inter‑sujet / échelle : chaque sujet a un niveau de baseline différent ; sans normalisation par baseline le modèle apprendra l’identité/échelle du sujet plutôt que la charge cognitive.
- Baselines et prétraitement nécessaires : les fichiers de baseline servent à calibrer (diviser) les features ; ils ne doivent pas être copiés tels quels comme exemples d’entraînement.
- Standardisation au bon moment : appliquer StandardScaler ou normalisation globale avant la séparation train/test provoque une fuite — il faut ajuster le scaler uniquement sur X_train (mettez-le dans un Pipeline).
- Métadonnées et labels : colonnes comme Participant, File, Window, Level ou Label ne doivent pas être utilisées comme features (elles fugueraient l’appariement avec la cible).
- Robustesse et généralisation : sans normalisation par sujet et validation adaptée (ex. LOSO), les performances sur sujets jamais vus seront surestimées.

In [1]:
from pathlib import Path

# TODO : adapter ce chemin à votre organisation locale.
BASE_PATH = Path("dataset")

EEG_FEATURE_DIR = BASE_PATH / "EEG_Features_10s"
LABEL_DIR = BASE_PATH / "Labels"
NORMALIZED_ROOT = BASE_PATH / "Normalized_Features_10s"
NORMALIZED_EEG_DIR = NORMALIZED_ROOT / "EEG"
LABELED_ROOT = BASE_PATH / "Normalized_Features_10s_With_Label"
LABELED_EEG_DIR = LABELED_ROOT / "EEG"

NORMALIZED_EEG_DIR.mkdir(parents=True, exist_ok=True)
LABELED_EEG_DIR.mkdir(parents=True, exist_ok=True)

METADATA_COLUMNS = ["Participant", "File", "Window", "Start_Time", "End_Time", "Channel"]

## 2. Normalisation des features EEG

À la fin du TD4, chaque fichier CSV contient des features EEG calculées sur des fenêtres de 10 secondes.

La normalisation doit suivre deux étapes :

### Étape 1 — Normalisation par la baseline du sujet

Pour chaque sujet, les fichiers de baseline servent à calculer une valeur moyenne de référence pour chaque feature :

$$
\mu_{baseline}^{(s,f)} = \frac{1}{N}\sum_{i=1}^{N} x_i^{(s,f)}
$$

où :

- $s$ désigne le sujet ;
- $f$ désigne la feature ;
- $x_i^{(s,f)}$ désigne la valeur de la feature pendant la baseline.

Chaque valeur de feature dans les fichiers de tâche est ensuite divisée par la moyenne de baseline correspondante :

$$
x_{norm}^{(s,f)} = \frac{ x^{(s,f)} }{ \mu_{baseline}^{(s,f)} }
$$

### Étape 2 — Standardisation z-score

On applique ensuite une standardisation :

$$
z = \frac{x - \mu}{\sigma}
$$

Cela permet d’obtenir des features centrées et réduites. Cette étape devra toutefois être réalisée plus loin dans le pipeline, après la séparation des données entre les ensembles d’entraînement et de test (voir section 8 ci-dessous).

## Question

Quel est l'intérêt de la normalisation par baseline dans des signaux physiologiques ?

### Réponse 

- Réduit la variance inter‑sujet : chaque sujet a un niveau physiologique (échelle) différent ; diviser par la moyenne de baseline évite que le modèle apprenne l’identité du sujet.
- Supprime les décalages de niveau (offsets) liés au matériel ou à l’enregistrement.
- Atténue la non‑stationnarité : compense les dérives lentes du signal entre la baseline et la tâche.
- Améliore la comparabilité des features entre conditions et facilite la détection d’effets liés à la charge cognitive plutôt qu’à l’échelle du signal.
-  Favorise la généralisation et une évaluation réaliste (surtout en LOSO).

In [2]:
import pandas as pd

def get_feature_columns(df, metadata_columns=METADATA_COLUMNS):
    """
    Retourne les colonnes numériques correspondant aux features.
    Les colonnes de métadonnées ne doivent pas être normalisées.
    """
    num_cols = df.select_dtypes(include=['number']).columns.tolist()
    feature_cols = [c for c in num_cols if c not in metadata_columns]
    return feature_cols

def compute_baseline_averages(feature_dir):
    """
    Calcule, pour chaque Participant, la moyenne de baseline de chaque feature.

    Indications :
    - parcourir les fichiers CSV de feature_dir ;
    - garder uniquement les fichiers dont le nom contient 'baseline' ;
    - lire chaque fichier avec pandas.read_csv ;
    - identifier le Participant avec df['Participant'].iloc[0] ;
    - calculer la moyenne de chaque feature ;
    """
    baseline_avgs = {}
    accum = {}
    feature_dir = Path(feature_dir)

    for csv_path in feature_dir.rglob('*.csv'):
        if 'baseline' not in csv_path.name.lower():
            continue
        try:
            df = pd.read_csv(csv_path)
        except Exception:
            continue
        if 'Participant' not in df.columns:
            continue

        pid = str(df['Participant'].iloc[0])
        feat_cols = get_feature_columns(df)
        if pid not in accum:
            accum[pid] = []
        if feat_cols:
            accum[pid].append(df[feat_cols])

    for pid, frames in accum.items():
        if not frames:
            continue
        allf = pd.concat(frames, ignore_index=True)
        baseline_avgs[pid] = allf.mean().to_dict()

    return baseline_avgs

def normalize_by_baseline(df, participant_id, baseline_avgs):
    """
    Divise chaque feature par sa moyenne de baseline pour le sujet considéré.
    """
    if participant_id not in baseline_avgs:
        raise KeyError(f"Aucune baseline disponible pour le participant {participant_id}")

    df_norm = df.copy()
    participant_avg = pd.Series(baseline_avgs[participant_id])
    feature_cols = [c for c in get_feature_columns(df_norm) if c in participant_avg.index]

    if feature_cols:
        df_norm.loc[:, feature_cols] = df_norm.loc[:, feature_cols].div(participant_avg[feature_cols])

    return df_norm

def run_eeg_normalization():
    """
    Génère les fichiers du dossier :
    Normalized_Features_10s/EEG
    à partir du dossier :
    EEG_Features_10s
    """
    baseline_avgs = compute_baseline_averages(EEG_FEATURE_DIR)

    for csv_path in EEG_FEATURE_DIR.rglob('*.csv'):
        if 'baseline' in csv_path.name.lower():
            continue

        try:
            df = pd.read_csv(csv_path)
        except Exception:
            continue

        if 'Participant' not in df.columns:
            continue

        participant_id = str(df['Participant'].iloc[0])
        if participant_id not in baseline_avgs:
            continue

        df_norm = normalize_by_baseline(df, participant_id, baseline_avgs)
        output_path = NORMALIZED_EEG_DIR / f"norm_{csv_path.stem}.csv"
        df_norm.to_csv(output_path, index=False)


In [3]:
run_eeg_normalization()

## 3. Vérification du dossier `Normalized_Features_10s/EEG`

Après exécution de la normalisation, vérifiez que le dossier contient bien des fichiers `norm_*.csv`.

## Question

Pourquoi les fichiers de baseline ne sont-ils pas copiés dans le dossier normalisé final ?

### Réponse

Les fichiers de baseline ont un rôle de **référence de calibration**, pas de données d'apprentissage. Leur seule utilité est de fournir la moyenne $\mu_{baseline}^{(s,f)}$ qui sert à normaliser les fichiers de tâche. Une fois cette normalisation effectuée, les fenêtres de baseline n'ont pas de label PAAS associé (le sujet est au repos, sans charge cognitive induite) : les inclure dans le dataset supervisé introduirait des exemples non étiquetables qui ne correspondent à aucune condition de charge cognitive. Elles sont donc consommées pendant la normalisation puis exclues du pipeline d'apprentissage.

In [4]:
# Vérification du dossier Normalized_Features_10s/EEG
import pandas as pd

norm_files = sorted(NORMALIZED_EEG_DIR.glob("norm_*.csv"))

if not norm_files:
    print(f"Aucun fichier norm_*.csv trouvé dans {NORMALIZED_EEG_DIR.resolve()}")
    print("Lance d'abord run_eeg_normalization().")
else:
    print(f"{len(norm_files)} fichier(s) dans {NORMALIZED_EEG_DIR}\n")
    print(f"{'Fichier':<45}  {'Lignes':>6}  {'Features':>8}  {'NaN':>5}  {'Participants'}")
    print("-" * 85)

    for f in norm_files:
        df_v = pd.read_csv(f)
        meta = [c for c in df_v.columns if c in METADATA_COLUMNS]
        feat_cols = [c for c in df_v.select_dtypes(include='number').columns if c not in meta]
        n_nan = int(df_v[feat_cols].isna().sum().sum())
        participants = df_v["Participant"].unique().tolist() if "Participant" in df_v.columns else ["?"]
        print(f"  {f.name:<43}  {len(df_v):>6}  {len(feat_cols):>8}  {n_nan:>5}  {participants}")

    # Aperçu du premier fichier
    df_sample = pd.read_csv(norm_files[0])
    print(f"\nAperçu — {norm_files[0].name} :")
    print(df_sample.head(3).to_string(index=False))

30 fichier(s) dans dataset/Normalized_Features_10s/EEG

Fichier                                        Lignes  Features    NaN  Participants
-------------------------------------------------------------------------------------
  norm_1030_eeg_features.csv                     1068        40      0  [1030]
  norm_1105_eeg_features.csv                     1068        40      0  [1105]
  norm_1106_eeg_features.csv                     1080        40      0  [1106]
  norm_1241_eeg_features.csv                     1080        40      0  [1241]
  norm_1271_eeg_features.csv                     1004        40      0  [1271]
  norm_1314_eeg_features.csv                      992        40      0  [1314]
  norm_1323_eeg_features.csv                     1080        40      0  [1323]
  norm_1337_eeg_features.csv                     1080        40      0  [1337]
  norm_1372_eeg_features.csv                     1080        40      0  [1372]
  norm_1417_eeg_features.csv                     1080        4

## 4. Ajout des colonnes `Level` et `Label`

Les fichiers normalisés ne contiennent pas encore la cible d'apprentissage.

Il faut maintenant associer chaque fenêtre de 10 secondes à son score PAAS.

Les labels sont stockés dans le dossier :

```text
Labels/
```

Chaque fichier de labels correspond à un sujet, par exemple :

```text
Labels/ID1.csv
Labels/ID2.csv
...
```

Dans ces fichiers, on suppose une structure du type :

| time | lvl_1 | lvl_2 | ... | lvl_9 |
|---:|---:|---:|---|---:|
| 10 | 2 | 3 | ... | 5 |
| 20 | 2 | 4 | ... | 6 |
| ... | ... | ... | ... | ... |

Pour une fenêtre d'indice `Window`, le temps associé est :

$$
time = (Window + 1) \times 10
$$

Le niveau du scénario est extrait du nom du fichier avec une expression régulière :

```text
level_1 → Level = 1
level_2 → Level = 2
...
level_9 → Level = 9
```

Le score PAAS est ensuite récupéré dans la colonne :

```text
lvl_<Level>
```

Exemple : si `Level = 4`, on lit la colonne `lvl_4`.


In [ ]:
def extract_level_from_filename(file_name):
    """
    Extrait le niveau de scénario à partir du nom de fichier.

    Exemple :
    filtered_level_3.csv → 3
    norm_filtered_level_8.csv → 8
    """
    # TODO : utiliser re.search.


def get_label_for_row(row, labels_df):
    """
    Retourne le score PAAS correspondant à une ligne de features.

    Indications :
    - récupérer Window ;
    - calculer time_stamp = (Window + 1) * 10 ;
    - récupérer Level ;
    - construire label_col = f'lvl_{Level}' ;
    - chercher dans labels_df la ligne où labels_df['time'] == time_stamp ;
    - retourner la valeur de label_col.
    """
    # TODO : compléter.


def attach_labels_eeg():
    """
    Génère les fichiers du dossier :
    Normalized_Features_10s_With_Label/EEG

    Chaque fichier de sortie doit contenir deux nouvelles colonnes :
    - Level
    - Label
    """
    # TODO : parcourir les fichiers normalisés.
    # TODO : identifier le Participant.
    # TODO : charger Labels/<Participant>.csv.
    # TODO : ajouter Level.
    # TODO : ajouter Label.
    # TODO : supprimer les lignes sans Label.
    # TODO : sauvegarder dans LABELED_EEG_DIR.

In [ ]:
attach_labels_eeg()

## 5. Vérification du dossier `Normalized_Features_10s_With_Label/EEG`

Le dossier final doit contenir des fichiers CSV avec au moins :

- les métadonnées : `Participant`, `File`, `Window`, `Channel`, `Start_Time` et `End_Time` ;
- les features EEG normalisées ;
- la colonne `Level` ;
- la colonne `Label`.

## Question

Quelle est la différence entre `Level` et `Label` dans ce TP ? Pourquoi faut-il ajouter à la fois `Level` et `Label` ?

### Réponse

- Level est le numéro du scénario de conduite (1 à 9), extrait du nom du fichier. C'est un identifiant de condition expérimentale, pas une mesure de charge cognitive.

- Label est le score PAAS associé à cette fenêtre temporelle pour ce niveau, issu du fichier de labels. C'est la cible d'apprentissage, une mesure subjective de la charge cognitive.

On ajoute les deux car Level est nécessaire pour aller chercher la bonne colonne (lvl_<Level>) dans le fichier de labels, et il peut servir de variable de traçabilité. Label est ce qu'on cherche à prédire.

In [ ]:
# Vérification du dossier Normalized_Features_10s_With_Label/EEG


## 6. Construction du dataset EEG supervisé

Une fois les fichiers normalisés et labellisés générés, on peut les concaténer pour construire un tableau unique.

Chaque ligne représente une fenêtre EEG de 10 secondes pour un canal.

On construit ensuite deux problèmes possibles :

### Classification binaire

| Score PAAS | Classe |
|---:|---|
| 1 à 4 | faible |
| 5 à 9 | élevée |

### Classification ternaire, extension

| Score PAAS | Classe |
|---:|---|
| 1 à 3 | faible |
| 4 à 6 | moyenne |
| 7 à 9 | élevée |

Dans ce TP, l'objectif principal est la classification binaire.

In [ ]:
def load_labeled_eeg_dataset():
    """
    Concatène tous les fichiers CSV du dossier Normalized_Features_10s_With_Label/EEG.
    """
    # TODO : parcourir LABELED_EEG_DIR, lire les CSV, concaténer avec pd.concat.


df = load_labeled_eeg_dataset()
print(df.shape)
df.head()

In [ ]:
# Création des cibles de classification.
df["Label_Binary"] = ...

# Extension ternaire éventuelle.
df["Label_Ternary"] =...

## 7. Préparation de la matrice d'apprentissage

On doit séparer :

- les métadonnées ;
- les features numériques EEG ;
- la cible d'apprentissage.

## Question

Pourquoi ne faut-il pas inclure `Participant`, `File`, `Window`, `Level` ou `Label` dans les features du modèle ?

### Réponse 

...

In [ ]:
# Préparation des données d’entraînement

print("Nombre d'exemples :", ...)
print("Nombre de features :", ...)
print("Exemples de features :", ..)

## 8. Classification EEG — premiers modèles

On teste plusieurs modèles classiques :

- LDA ;
- SVM ;
- Random Forest ;
- KNN ;
- Naive Bayes ;
- Decision Tree ;
- AdaBoost ;
- MLP.

La normalisation `StandardScaler` est placée dans le `sklearn.pipeline.Pipeline` pour éviter une fuite de données entre apprentissage et test. Il faut ajuster le `StandardScaler` uniquement sur les données d’entraînement :

`scaler.fit_transform(X_train)`

Puis appliquer la transformation aux données de test avec :

`scaler.transform(X_test)`

## 9. Évaluation par validation croisée et par sujet

Deux évaluations sont demandées :

### 10-fold cross-validation

Les segments sont répartis en 10 folds stratifiés. Cette évaluation est utile pour comparer les modèles, mais elle peut mélanger les sujets entre apprentissage et test.

### Leave-One-Subject-Out, LOSO

Un sujet est laissé de côté pour le test, tandis que le modèle est entraîné sur les autres sujets. Cette stratégie d’évaluation est plus réaliste, car elle permet de tester la capacité de généralisation du modèle sur un conducteur jamais vu auparavant. L’opération est ensuite répétée sur l’ensemble des sujets disponibles afin d’obtenir une évaluation plus robuste.

## Question

Pourquoi le LOSO est-il souvent plus difficile que le 10-fold classique ?

### Réponse 

...

## 10. Interprétation et discussion

Répondez aux questions suivantes dans le notebook :

1. Quel modèle obtient le meilleur F1-score en 10-fold ?
2. Quel modèle obtient le meilleur F1-score en LOSO ?
3. Les performances chutent-elles en LOSO ? Pourquoi ?
4. Les classes sont-elles équilibrées ?
5. Les résultats obtenus avec EEG seul vous semblent-ils suffisants pour une application réelle ?
6. Quelles limites voyez-vous à l'utilisation des labels subjectifs PAAS ?
7. Quelles améliorations proposeriez-vous ?

## 11. Mini-système d'adaptation

À partir de la prédiction du modèle, on peut simuler une décision d'adaptation.

Exemple :

| Prédiction | Décision |
|---|---|
| charge faible | interface normale |
| charge élevée | simplification de l'interface |
| charge élevée persistante | alerte conducteur |

## Question 

Pourquoi faut-il être prudent avant de déclencher une alerte sur une seule prédiction ?

### Réponse 

...

In [ ]:
def decision_system(...):
    """
    Transforme les prédictions en décision d'adaptation.
    
    """




## 12. Extension optionnelle — vers la multimodalité

Le cœur du TP est volontairement limité à l'EEG.

Une extension possible consiste à reproduire les mêmes étapes pour les autres modalités :

```text
ECG_Features_10s → Normalized_Features_10s/ECG → Normalized_Features_10s_With_Label/ECG
EDA_Features_10s → Normalized_Features_10s/EDA → Normalized_Features_10s_With_Label/EDA
Gaze_Features_10s → Normalized_Features_10s/Gaze → Normalized_Features_10s_With_Label/Gaze
```

Puis à fusionner les features :

```text
EEG + ECG
EEG + EDA
EEG + Gaze
EEG + ECG + EDA + Gaze
```

La fusion la plus simple est une concaténation des colonnes de features pour des fenêtres correspondant au même sujet, au même niveau et au même indice de fenêtre.

## Question

Pourquoi la multimodalité peut-elle améliorer la détection de la charge cognitive ?

### Réponse 

...